# 22 · Cases by period, checked against pandas

## What this checks

**Cases by period** counts dated records by day, by week or by month. This is the epidemic curve. This check covers
release 0.10.0.

A record counted in the wrong period moves the peak, or hides a gap. So it is worth checking.

## How it is checked

We count the same records again with **pandas**, Python's library for tables. Pandas groups dates into periods with
its own method, called resampling. Fieldwork uses its own calendar arithmetic.

This is a **genuinely independent** check. The two are different programs of different design, and share no code.
That is unlike check 21 next door, where both sides call the same library.

The data are Table 1 from Snow's 1855 report: deaths by date, in 43 daily rows. This is the epidemic curve the John
Snow example draws. They are counted by day, by week and by month.

## What should happen

For each of day, week and month:

1. **The periods** are the same, in the same order, exactly. Each period is named by its first day, or by its month.
2. **The count in each period** is the same, exactly. Each row carries a number of deaths, so the count is a sum.
3. **The total** is the same, exactly.
4. **No period is missing** between the first and the last, including periods with no deaths.
5. **The dated and undated records** are counted the same, exactly.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_22_case_series import run

report = run(Path.cwd() / "fixtures" / "case-series-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], f"over {report['rows']} rows "
      f"({report['datedRows']} dated, {report['undatedRows']} undated)")
print()
for period in report["periods"]:
    print(f"by {period['period']:<6} resample({period['resampleRule']})")
    print(f"  {period['binsReference']:>3} bins, {period['emptyBins']} with no deaths, "
          f"total {period['totalReference']:.0f}")
    print(f"  {period['firstKey']} to {period['lastKey']}, peak {period['peak'][0]} "
          f"at {period['peak'][1]:.0f}")
    print()

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.10.0:
**Agrees**.

All 43 rows had dates. Every period matched exactly, in order, with the same counts and the same total of 616 deaths:

- **By day**, 43 periods, 12 of them with no deaths.
- **By week**, 7 periods.
- **By month**, 2 periods.

Running the cell above repeats the check on the same data.

## The week was the dangerous case, and it caught the check

This check's own notes named the week as the rule most likely to go wrong. **The first version of the check got it
wrong anyway.**

Fieldwork names a week by its Monday, as the ISO calendar does. The check first asked pandas for weeks named by the
day before they start, which is a **Sunday**. So every week came out one day early. Yet every count looked entirely
plausible: 7 weeks, a total of 616, the peak in the right week. Nothing about the shape of the curve looked wrong.

Only comparing the **names of the periods** caught it. Predicting the hazard did not prevent walking into it. The
cell below shows three plausible ways to ask pandas for weeks. They give three different answers, and only one is
right. The counts alone would not tell them apart.

In [ ]:
import pandas as pd

fixture = json.loads((Path.cwd() / "fixtures" / "case-series-soho-001.json").read_text())
rows = fixture["input"]["rows"]
frame = pd.DataFrame({"date": pd.to_datetime([r["date"] for r in rows]),
                      "deaths": [r["deaths"] for r in rows]}).set_index("date")

fieldwork = [(b["key"], b["count"]) for b in fixture["output"]["week"]["bins"]]
print(f"{'rule':<34} {'first three bins'}")
for rule, options in [("W-MON, label=left, closed=left", dict(label="left", closed="left")),
                      ("W-SUN, label=left, closed=left", dict(label="left", closed="left")),
                      ("W-MON, default labelling", {})]:
    frequency = rule.split(",")[0]
    binned = frame["deaths"].resample(frequency, **options).sum()
    first = [(t.strftime("%Y-%m-%d"), int(v)) for t, v in binned.items()][:3]
    print(f"{rule:<34} {first}")
print(f"{'fieldwork':<34} {fieldwork[:3]}")
print()
print("Three plausible mappings, three different answers, one of them right. The counts alone")
print("would not have told them apart: it is the keys that differ.")

## What this does not show

- That the dates are right.
- That a curve by date of **death** says what a reader will take it to say. Snow's table records deaths by the date
  of death. The shape by date of first symptoms would be earlier, by the course of the illness.
- Anything about onset dates. The widget carries the kind of date it was given, so a chart cannot quietly relabel it.
  No check can supply onset dates that nobody recorded.